# Task 2: EDA & Data Pipeline — Wikipedia Next-Click Prediction

Given `current_article_id` and `target_article_id`, predict `next_article_id`.
**Metric**: Accuracy (exact match)

This notebook generates and caches all features needed for downstream models:
- Link graph from Wikispeedia
- Text embeddings (sentence-transformers)
- Visual embeddings (CLIP from screenshots)
- Wikipedia2Vec embeddings (graph-aware)
- Category encoding

In [ ]:
import io, pickle, tarfile, urllib.request, urllib.parse, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from PIL import Image
import torch
from sentence_transformers import SentenceTransformer

%matplotlib inline
sns.set_style('whitegrid')

ROOT = Path.cwd()
while not (ROOT / 'dataset-task2').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / 'dataset-task2'
CACHE = ROOT / '.cache'
CACHE.mkdir(exist_ok=True)
print(f'ROOT: {ROOT}')
print(f'DATA: {DATA}')

## 1. Basic Data Loading

In [ ]:
articles = pd.read_csv(DATA / 'articles.csv')
categories = pd.read_csv(DATA / 'categories.csv')
train = pd.read_csv(DATA / 'states_train.csv')
test = pd.read_csv(DATA / 'states_test.csv')
sample_sub = pd.read_csv(DATA / 'sample_submission.csv')

print(f'Articles: {articles.shape}')
print(f'Categories: {categories.shape}')
print(f'Train states: {train.shape}')
print(f'Test states: {test.shape}')
print(f'Sample submission: {sample_sub.shape}')
train.head()

In [ ]:
all_ids = set(articles['article_id'])
print(f'Article IDs: {articles.article_id.min()} to {articles.article_id.max()}')
print(f'Unique current articles: {train.current_article_id.nunique()}')
print(f'Unique target articles: {train.target_article_id.nunique()}')
print(f'Unique next articles: {train.next_article_id.nunique()}')
print(f'All next_article_ids valid: {train.next_article_id.isin(all_ids).all()}')
print(f'All current_article_ids valid: {train.current_article_id.isin(all_ids).all()}')
print(f'All target_article_ids valid: {train.target_article_id.isin(all_ids).all()}')

In [ ]:
cat_counts = categories.groupby('article_id').size()
print(f'Articles with >=1 category: {len(cat_counts)}')
print(f'Cats per article - mean: {cat_counts.mean():.1f}, max: {cat_counts.max()}')
cat_counts.hist(bins=20, figsize=(8, 4))
plt.title('Distribution of categories per article')
plt.xlabel('Number of categories'); plt.ylabel('Count')
plt.show()

## 2. Link Graph from Wikispeedia

The article set matches the Wikispeedia dataset (4604 articles, 119,880 edges). Auto-downloads from SNAP if not cached.

In [ ]:
def load_adjacency(arts, cache_dir):
    cache_path = cache_dir / 'wikispeedia_adj.pkl'
    if cache_path.exists():
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    title_to_id = dict(zip(arts['title'].str.strip(), arts['article_id']))
    url = 'https://snap.stanford.edu/data/wikispeedia/wikispeedia_paths-and-graph.tar.gz'
    print(f'Downloading Wikispeedia from {url} ...')
    with urllib.request.urlopen(url) as resp:
        with tarfile.open(fileobj=io.BytesIO(resp.read()), mode='r:gz') as tar:
            f = tar.extractfile('wikispeedia_paths-and-graph/links.tsv')
            content = f.read()
    links = pd.read_csv(io.BytesIO(content), sep='\t', skiprows=14,
                        header=None, names=['source', 'target'])
    links['sid'] = links['source'].apply(
        lambda x: urllib.parse.unquote(x).replace('_', ' ').strip()).map(title_to_id)
    links['tid'] = links['target'].apply(
        lambda x: urllib.parse.unquote(x).replace('_', ' ').strip()).map(title_to_id)
    links = links.dropna(subset=['sid', 'tid'])
    adj = {i: [] for i in range(len(arts))}
    for _, r in links.iterrows():
        adj[int(r['sid'])].append(int(r['tid']))
    adj = {k: list(set(v)) for k, v in adj.items()}
    with open(cache_path, 'wb') as f:
        pickle.dump(adj, f)
    return adj

adj = load_adjacency(articles, CACHE)
degrees = [len(adj[i]) for i in range(len(adj))]

print(f'Link graph: {len(adj)} nodes, {sum(degrees):,} edges')
print(f'Avg out-degree: {np.mean(degrees):.1f}')
print(f'Min out-degree: {min(degrees)}, Max out-degree: {max(degrees)}')
print(f'Nodes with 0 out-links: {sum(1 for d in degrees if d == 0)}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(degrees, bins=50, log=True)
axes[0].set_title('Out-degree distribution (log)')
axes[0].set_xlabel('Degree'); axes[0].set_ylabel('Count')
axes[1].hist(degrees, bins=50, cumulative=True, density=True, histtype='step')
axes[1].set_title('Cumulative out-degree distribution')
axes[1].set_xlabel('Degree')
plt.tight_layout(); plt.show()

### Validate: Is the next_article always a link on the current page?

In [ ]:
valid = train.apply(lambda r: r['next_article_id'] in adj.get(r['current_article_id'], []), axis=1)
print(f'next_article is a link on current page: {valid.mean()*100:.2f}% ({valid.sum()}/{len(train)})')
invalid = train[~valid]
if len(invalid) > 0:
    print(f'{len(invalid)} invalid examples:')
    display(invalid.head(10))

### BFS distance from current to target

In [ ]:
def bfs_dist(adj, start, target, max_dist=10):
    if start == target:
        return 0
    visited = {start}
    queue = [(start, 0)]
    while queue:
        node, d = queue.pop(0)
        if node == target:
            return d
        if d >= max_dist:
            continue
        for nb in adj.get(node, []):
            if nb not in visited:
                visited.add(nb)
                queue.append((nb, d + 1))
    return max_dist

sample = train.sample(min(1000, len(train)), random_state=42)
dists = [bfs_dist(adj, r['current_article_id'], r['target_article_id'], 10)
         for _, r in sample.iterrows()]

plt.figure(figsize=(8, 4))
plt.hist(dists, bins=range(0, 12), align='left', rwidth=0.8)
plt.title('BFS distance from current to target (sampled 1000)')
plt.xlabel('Distance (capped at 10)'); plt.xticks(range(0, 11))
plt.show()
print(f'Same article (dist=0): {sum(1 for d in dists if d==0)}')
print(f'Direct link (dist=1): {sum(1 for d in dists if d==1)}')
print(f'2-3 hops: {sum(1 for d in dists if 2<=d<=3)}')
print(f'4-10 hops: {sum(1 for d in dists if 4<=d<=10)}')

## 3. Text Embeddings (sentence-transformers)

384-d embeddings from article titles using `all-MiniLM-L6-v2`.

In [ ]:
embed_cache = CACHE / 'article_embeddings.npy'
if embed_cache.exists():
    text_embeddings = np.load(embed_cache)
    print(f'Loaded cached: {text_embeddings.shape}')
else:
    model = SentenceTransformer('all-MiniLM-L6-v2')
    text_embeddings = model.encode(articles['title'].tolist(), show_progress_bar=True, batch_size=64)
    np.save(embed_cache, text_embeddings)
    print(f'Generated: {text_embeddings.shape}')

In [ ]:
# Quick sanity: title-similarity baseline
def title_sim_acc(train_df, adj, emb, n=500):
    df = train_df.sample(min(n, len(train_df)), random_state=42)
    correct = total = 0
    for _, r in df.iterrows():
        links = adj.get(r['current_article_id'], [])
        if not links:
            continue
        scores = emb[links] @ emb[r['target_article_id']]
        if links[scores.argmax()] == r['next_article_id']:
            correct += 1
        total += 1
    return correct / total if total else 0

print(f'Title-similarity baseline (n=500): {title_sim_acc(train, adj, text_embeddings)*100:.2f}%')

## 4. Visual Embeddings (CLIP from Screenshots)

512-d embeddings from article screenshots using `openai/clip-vit-base-patch32`. Runs on GPU if available.

In [ ]:
clip_cache = CACHE / 'clip_embeddings.npy'

if clip_cache.exists():
    clip_embeddings = np.load(clip_cache)
    print(f'Loaded cached CLIP: {clip_embeddings.shape}')
else:
    from transformers import CLIPProcessor, CLIPModel
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device).eval()
    processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
    
    embs = []
    screenshot_dir = DATA / 'screenshots'
    missing = 0
    for aid in tqdm(range(len(articles)), desc='CLIP'):
        fname = screenshot_dir / f'{aid}.png'
        if not fname.exists():
            embs.append(np.zeros(512, dtype=np.float32))
            missing += 1
            continue
        image = Image.open(fname).convert('RGB')
        inputs = processor(images=image, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            emb = model.get_image_features(**inputs).pooler_output.cpu().numpy().flatten()
        embs.append(emb)
    clip_embeddings = np.array(embs, dtype=np.float32)
    np.save(clip_cache, clip_embeddings)
    print(f'Generated CLIP: {clip_embeddings.shape}, missing: {missing}')

## 5. Wikipedia2Vec Embeddings

Graph-aware embeddings from Wikipedia link graph + text. Uses gensim (text format) since the binary model URL returns 403.

In [ ]:
wiki2vec_cache = CACHE / 'wiki2vec_embeddings.npy'

if wiki2vec_cache.exists():
    wiki2vec_embeddings = np.load(wiki2vec_cache)
    print(f'Loaded cached Wikipedia2Vec: {wiki2vec_embeddings.shape}')
else:
    import bz2, gensim
    url = 'https://wikipedia2vec.s3-ap-northeast-1.amazonaws.com/models/en/2018-04-20/enwiki_20180420_100d.txt.bz2'
    bz2_file = CACHE / 'enwiki_20180420_100d.txt.bz2'
    txt_file = CACHE / 'enwiki_20180420_100d.txt'
    if not bz2_file.exists():
        print(f'Downloading Wikipedia2Vec model (~927MB) ...')
        import urllib.request
        urllib.request.urlretrieve(url, bz2_file)
    if not txt_file.exists():
        print('Decompressing ...')
        with open(txt_file, 'wb') as out:
            out.write(bz2.decompress(bz2_file.read_bytes()))
    print('Loading word vectors ...')
    w2v = gensim.models.KeyedVectors.load_word2vec_format(str(txt_file))
    
    embs = []
    for _, title in tqdm(articles['title'].items(), total=len(articles), desc='Wiki2Vec'):
        words = [w.lower() for w in title.split() if w.lower() in w2v]
        if words:
            embs.append(np.mean([w2v[w] for w in words], axis=0))
        else:
            embs.append(np.zeros(100, dtype=np.float32))
    wiki2vec_embeddings = np.array(embs, dtype=np.float32)
    np.save(wiki2vec_cache, wiki2vec_embeddings)
    print(f'Generated Wikipedia2Vec: {wiki2vec_embeddings.shape}')

## 6. Category Encoding

Multi-hot encoding of 129 unique categories.

In [ ]:
all_categories = sorted(categories['category'].unique())
cat_to_idx = {c: i for i, c in enumerate(all_categories)}
n_cats = len(all_categories)
print(f'Unique categories: {n_cats}')

cat_encoding = np.zeros((len(articles), n_cats), dtype=np.float32)
for _, row in categories.iterrows():
    cat_encoding[row['article_id'], cat_to_idx[row['category']]] = 1.0

top_cats = categories['category'].value_counts().head(15)
plt.figure(figsize=(10, 4))
top_cats.plot(kind='bar')
plt.title('Top 15 categories'); plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# Quick sanity: category overlap baseline
def cat_overlap_acc(train_df, adj, cat_enc, n=500):
    df = train_df.sample(min(n, len(train_df)), random_state=42)
    correct = total = 0
    for _, r in df.iterrows():
        links = adj.get(r['current_article_id'], [])
        if not links:
            continue
        scores = (cat_enc[links] * cat_enc[r['target_article_id']][None, :]).sum(axis=1)
        if links[scores.argmax()] == r['next_article_id']:
            correct += 1
        total += 1
    return correct / total if total else 0

print(f'Category overlap baseline (n=500): {cat_overlap_acc(train, adj, cat_encoding)*100:.2f}%')

## 7. Save All Features

Combined feature matrix: text (384) + CLIP (512) + wiki2vec (100) + category (129) = 1125 dims.

In [ ]:
combined = np.concatenate([text_embeddings, clip_embeddings, wiki2vec_embeddings, cat_encoding], axis=1)
print(f'Combined features: {combined.shape} (384 text + 512 CLIP + 100 wiki2vec + 129 cats)')

np.save(CACHE / 'text_embeddings_384.npy', text_embeddings)
np.save(CACHE / 'clip_embeddings_512.npy', clip_embeddings)
np.save(CACHE / 'wiki2vec_embeddings_100.npy', wiki2vec_embeddings)
np.save(CACHE / 'cat_encoding_129.npy', cat_encoding)
np.save(CACHE / 'combined_features_1125.npy', combined)

with open(CACHE / 'adj.pkl', 'wb') as f:
    pickle.dump(adj, f)

print('All features saved to .cache/')

## 8. Summary

In [ ]:
print('EDA complete.')
print(f'  Articles: {len(articles)}')
print(f'  Directed links: {sum(len(v) for v in adj.values()):,}')
print(f'  Train states: {len(train)}, Test states: {len(test)}')
print(f'  Text embedding: {text_embeddings.shape[1]}d (sentence-transformers)')
print(f'  CLIP embedding: {clip_embeddings.shape[1]}d (vit-base-patch32)')
print(f'  Wiki2Vec embedding: {wiki2vec_embeddings.shape[1]}d (enwiki_20180420)')
print(f'  Category encoding: {cat_encoding.shape[1]}d (multi-hot)')
print(f'  Combined features: 1125d')
print(f'  Title-sim baseline: {title_sim_acc(train, adj, text_embeddings)*100:.2f}%')
print(f'  Category baseline: {cat_overlap_acc(train, adj, cat_encoding)*100:.2f}%')